In [1]:
import numpy as np

encoder_input = np.load('encoder_input.npy')
decoder_input = np.load('decoder_input.npy')
decoder_output = np.load('decoder_output.npy')

In [2]:
import joblib

idx2word = joblib.load('idx2word.pkl')
word2idx = joblib.load('word2idx.pkl')

In [3]:
from sklearn.model_selection import train_test_split

encoder_input_train, encoder_input_test, decoder_input_train, decoder_input_test, decoder_output_train, decoder_output_test = train_test_split(encoder_input, decoder_input, decoder_output, test_size=0.2, random_state=42)

In [4]:
vocab_size = len(word2idx)
vocab_size

26629

In [5]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, Embedding

encoder_inputs = Input(shape=(None,))
encoder_embedding_layer = Embedding(vocab_size, 256)
encoder_embedded_inputs = encoder_embedding_layer(encoder_inputs)
encoder_gru_layer = GRU(256, return_state=True, return_sequences=True)
encoder_outputs, state_h = encoder_gru_layer(encoder_embedded_inputs)
encoder_states = state_h

In [6]:
decoder_inputs = Input(shape=(None,))
decoder_embedding_layer = Embedding(vocab_size, 256)
decoder_embedded_inputs = decoder_embedding_layer(decoder_inputs)
decoder_gru_layer = GRU(256, return_sequences=True, return_state=True)
decoder_outputs, _ = decoder_gru_layer(decoder_embedded_inputs, initial_state=encoder_states)

In [7]:
from tensorflow.keras.layers import Attention

attention = Attention()
context_vector = attention([decoder_outputs, encoder_outputs])

In [8]:
from tensorflow.keras.layers import Concatenate, Dense

decoder_combined_context = Concatenate(axis=-1)([decoder_outputs, context_vector])
decoder_dense = Dense(vocab_size, activation='softmax')
decoder_prediction = decoder_dense(decoder_combined_context)

In [9]:
model = Model([encoder_inputs, decoder_inputs], decoder_prediction)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │  6,817,024 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │  6,817,024 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, None,     │    394,752 │ embedding[0][0]   │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, None,     │    394,752 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ gru[0][1]         │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 256) │          0 │ gru_1[0][0],      │
│ (Attention)         │                   │            │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 512) │          0 │ gru_1[0][0],      │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 13,660,677 │ concatenate[0][0] │
│                     │ 26629)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 28,084,229 (107.13 MB)

 Trainable params: 28,084,229 (107.13 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
import numpy as np

decoder_output_train_expanded = np.expand_dims(decoder_output_train, -1)
decoder_output_test_expanded = np.expand_dims(decoder_output_test, -1)

In [11]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

In [12]:
history = model.fit(
    [encoder_input_train, decoder_input_train],
    decoder_output_train_expanded,
    validation_data=([encoder_input_test, decoder_input_test], decoder_output_test_expanded),
    batch_size=128,
    epochs=15,
    callbacks=[early_stop, reduce_lr],
)

Epoch 1/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 524s 414ms/step - accuracy: 0.7621 - loss: 1.7021 - val_accuracy: 0.7744 - val_loss: 1.4856 - learning_rate: 0.0010
Epoch 2/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 530s 424ms/step - accuracy: 0.7777 - loss: 1.4214 - val_accuracy: 0.7789 - val_loss: 1.4050 - learning_rate: 0.0010
Epoch 3/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 532s 425ms/step - accuracy: 0.7817 - loss: 1.3313 - val_accuracy: 0.7803 - val_loss: 1.3803 - learning_rate: 0.0010
Epoch 4/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 531s 425ms/step - accuracy: 0.7850 - loss: 1.2558 - val_accuracy: 0.7807 - val_loss: 1.3806 - learning_rate: 0.0010
Epoch 5/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 531s 424ms/step - accuracy: 0.7889 - loss: 1.1789 - val_accuracy: 0.7801 - val_loss: 1.3980 - learning_rate: 0.0010
Epoch 6/15
1251/1251 ━━━━━━━━━━━━━━━━━━━━ 531s 425ms/step - accuracy: 0.7972 - loss: 1.0650 - val_accuracy: 0.7790 - val_loss: 1.4213 - learning_rate: 5.0000e-04


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------- Encoder inference model ----------
# Now outputs BOTH the full sequence (for attention) AND the final state (to init decoder)

encoder_model = Model(encoder_inputs, [encoder_outputs, state_h])

In [17]:
decoder_state_input_h = Input(shape=(256,))

# Incoming encoder outputs (needed fresh each step for attention to look back at)
encoder_outputs_input = Input(shape=(None, 256))

# Single-token decoder input for inference
decoder_input_inference = Input(shape=(1,))
# Reuse trained embedding layer to embed the single token input
decoder_embedded_input_inference = decoder_embedding_layer(decoder_input_inference)

decoder_gru_outputs_inf, state_h_inf = decoder_gru_layer(
    decoder_embedded_input_inference, initial_state=decoder_state_input_h
)

In [24]:
# Attention: decoder's current output vs the passed-in encoder outputs
attention_inf = Attention()([decoder_gru_outputs_inf, encoder_outputs_input])

concat_inf = Concatenate(axis=-1)([decoder_gru_outputs_inf, attention_inf])

decoder_outputs_inf = decoder_dense(concat_inf)

decoder_model = Model(
    [decoder_input_inference, encoder_outputs_input, decoder_state_input_h],
    [decoder_outputs_inf, state_h_inf]
)

In [19]:
def sentence_to_ids(sentence, word2idx):
    return [word2idx.get(word, word2idx["<UNK>"]) for word in sentence.split()]

In [25]:
def generate_response(input_text, max_len=33):
    input_seq = sentence_to_ids(input_text, word2idx)
    input_seq = np.array([input_seq])
    input_seq = pad_sequences(input_seq, maxlen=33, padding="post")

    # Get encoder outputs AND initial state
    enc_outs, state_h_value = encoder_model.predict(input_seq, verbose=0)

    target_seq = np.array([[word2idx["<START>"]]])
    generated_words = []

    for _ in range(max_len):
        output_tokens, h = decoder_model.predict(
            [target_seq, enc_outs, state_h_value], verbose=0
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = idx2word.get(sampled_token_index, "<UNK>")

        if sampled_word == "<END>":
            break

        generated_words.append(sampled_word)

        target_seq = np.array([[sampled_token_index]])
        state_h_value = h

    return " ".join(generated_words)


In [21]:
test_inputs = [
    "how are you",
    "what is your name",
    "do you love me",
]

In [26]:
for text in test_inputs:
    print(f"You: {text}")
    print(f"Bot: {generate_response(text)}")
    print()

You: how are you
Bot: I don't know.

You: what is your name
Bot: <UNK>

You: do you love me
Bot: I don't know.



In [28]:
model.save("gru_model.h5")